In [17]:
import pandas as pd




In [18]:
class CSVReader:

    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n=5):
        if self.df is None:
            print("Please read the CSV file first.")
            return

        print(self.df.head(n))

In [19]:
reader = CSVReader("sample_data (2).csv")

In [20]:
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [21]:
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [22]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Column names:", df.columns.tolist())

Number of rows: 7
Number of columns: 7
Column names: ['id', 'name', 'age', 'height_cm', 'weight_kg', 'city', 'score']


## Exercise 2


In [23]:
import pandas as pd
from abc import ABC, abstractmethod

## Create the abstract class


In [24]:
class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

In [25]:
class DropMissing(MissingValueStrategy):

    def handle(self, df):
        return df.dropna().copy()

In [26]:
class FillMean(MissingValueStrategy):

    def handle(self, df):
        result = df.copy()

        numeric_columns = result.select_dtypes(
            include="number"
        ).columns

        result[numeric_columns] = result[
            numeric_columns
        ].fillna(result[numeric_columns].mean())

        return result

In [27]:
class FillMode(MissingValueStrategy):

    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            modes = result[column].mode(dropna=True)

            if not modes.empty:
                result[column] = result[column].fillna(
                    modes.iloc[0]
                )

        return result

In [28]:
class DataCleaner:

    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [29]:
print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


In [30]:
cleaner_drop = DataCleaner(DropMissing())

df_drop = cleaner_drop.clean(df)

print("After DropMissing:")
display(df_drop)

print("Rows before:", len(df))
print("Rows after:", len(df_drop))

After DropMissing:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


Rows before: 7
Rows after: 2


In [31]:
cleaner_mean = DataCleaner(FillMean())

df_mean = cleaner_mean.clean(df)

print("After FillMean:")
display(df_mean)

print("Missing values remaining:")
print(df_mean.isnull().sum())

After FillMean:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


Missing values remaining:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


In [32]:
cleaner_mode = DataCleaner(FillMode())

df_mode = cleaner_mode.clean(df)

print("After FillMode:")
display(df_mode)

print("Missing values remaining:")
print(df_mode.isnull().sum())

After FillMode:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


Missing values remaining:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


In [33]:
comparison = pd.DataFrame({
    "Original": df.isnull().sum(),
    "DropMissing": df_drop.isnull().sum(),
    "FillMean": df_mean.isnull().sum(),
    "FillMode": df_mode.isnull().sum()
})

display(comparison)

,Original,DropMissing,FillMean,FillMode
id,0,0,0,0
name,0,0,0,0
age,3,0,0,0
height_cm,1,0,0,0
weight_kg,1,0,0,0
city,0,0,0,0
score,2,0,0,0


## Exercise 3

In [34]:
import pandas as pd
import time
from functools import wraps
import logging

In [35]:
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting: {func.__name__}")

        result = func(*args, **kwargs)

        print(f"Finished: {func.__name__}")

        return result

    return wrapper

In [36]:
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()

        elapsed = end - start

        print(f"Execution time: {elapsed:.6f} seconds")

        return result

    return wrapper

In [37]:
class CSVReader:

    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n=5):
        if self.df is None:
            print("Please read the CSV file first.")
            return

        print(self.df.head(n))

In [38]:
reader = CSVReader("sample_data (2).csv")

df = reader.read()

reader.preview(5)

Starting: read
Execution time: 0.004066 seconds
Finished: read
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Exercise 4

In [39]:
from abc import ABC, abstractmethod
import pandas as pd
import re

class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass

In [40]:
class NormalizeColumns(DataTransform):
    def apply(self, df):
        result = df.copy()

        result.columns = [
            re.sub(r"_+", "_",
                   re.sub(r"[^a-zA-Z0-9]+", "_", str(col).strip().lower())
            ).strip("_")
            for col in result.columns
        ]

        return result

In [41]:
class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().copy()

In [42]:
class StandardizeText(DataTransform):
    def apply(self, df):
        result = df.copy()

        text_columns = result.select_dtypes(
            include=["object", "string"]
        ).columns

        for column in text_columns:
            result[column] = result[column].apply(
                lambda value: re.sub(r"\s+", " ", value.strip())
                if isinstance(value, str) else value
            )

        return result

In [43]:
class TransformFactory:
    @staticmethod
    def get_transform(transform_type):
        transforms = {
            "normalize": NormalizeColumns,
            "duplicates": RemoveDuplicates,
            "text": StandardizeText
        }

        transform_class = transforms.get(transform_type.lower())

        if transform_class is None:
            raise ValueError(
                f"Unknown transformation: {transform_type}"
            )

        return transform_class()

In [44]:
# Create transformation objects using the factory
normalize = TransformFactory.get_transform("normalize")
remove_duplicates = TransformFactory.get_transform("duplicates")
standardize_text = TransformFactory.get_transform("text")

# Apply transformations
df_normalized = normalize.apply(df)
df_no_duplicates = remove_duplicates.apply(df)
df_standardized = standardize_text.apply(df)

# Display results
print("Original columns:")
print(df.columns.tolist())

print("\nNormalized columns:")
print(df_normalized.columns.tolist())

print("\nOriginal rows:", len(df))
print("Rows after removing duplicates:", len(df_no_duplicates))

print("\nStandardized text preview:")
display(df_standardized.head())

Original columns:
['id', 'name', 'age', 'height_cm', 'weight_kg', 'city', 'score']

Normalized columns:
['id', 'name', 'age', 'height_cm', 'weight_kg', 'city', 'score']

Original rows: 7
Rows after removing duplicates: 7

Standardized text preview:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


## Exercise 5

In [45]:
from abc import ABC, abstractmethod

class DataPipeline(ABC):

    def run(self):
        """Template Method: defines the workflow."""
        self.load()
        self.clean()
        self.transform()
        self.save()

        print("Cleaning pipeline completed successfully!")
        return self.df

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass

In [46]:
class CSVDataPipeline(DataPipeline):

    def __init__(
        self,
        input_path,
        output_path,
        missing_strategy,
        transform_types=None
    ):
        self.input_path = input_path
        self.output_path = output_path
        self.missing_strategy = missing_strategy
        self.transform_types = transform_types or []
        self.df = None

    def load(self):
        reader = CSVReader(self.input_path)
        self.df = reader.read()
        print("1. Data loaded:", self.df.shape)

    def clean(self):
        cleaner = DataCleaner(self.missing_strategy)
        self.df = cleaner.clean(self.df)
        print("2. Missing values handled:", self.df.shape)

    def transform(self):
        for transform_type in self.transform_types:
            transform = TransformFactory.get_transform(
                transform_type
            )
            self.df = transform.apply(self.df)

        print("3. Transformations completed:", self.df.shape)

    def save(self):
        self.df.to_csv(self.output_path, index=False)
        print("4. Data saved to:", self.output_path)

In [48]:
pipeline = CSVDataPipeline(
    input_path="sample_data (2).csv",
    output_path="cleaned_data.csv",
    missing_strategy=FillMode(),
    transform_types=[
        "normalize",
        "text",
        "duplicates"
    ]
)

df_cleaned = pipeline.run()

display(df_cleaned.head())

Starting: read
Execution time: 0.004283 seconds
Finished: read
1. Data loaded: (7, 7)
2. Missing values handled: (7, 7)
3. Transformations completed: (7, 7)
4. Data saved to: cleaned_data.csv
Cleaning pipeline completed successfully!


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


In [49]:
df_check = pd.read_csv("cleaned_data.csv")

print("Saved file shape:", df_check.shape)
print("Missing values:")
print(df_check.isnull().sum())

display(df_check.head())

Saved file shape: (7, 7)
Missing values:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
